In [1]:
# ── Imports ───────────────────────────────────────────────────────────────
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datetime
import warnings

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    warnings.warn('wandb not installed — logging disabled.')

from env import FoodEnv
from agent import PPOAgent

now           = datetime.datetime.now()
curr_date_time = 'd_' + now.strftime('%d_%m_%Y') + '_t_' + now.strftime('%H_%M_%S')
print('Run timestamp:', curr_date_time)

Run timestamp: d_24_05_2026_t_22_15_34


d:\heisen\BPCL\Code\RL\FoodRL\utils.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# ── WandB initialisation ──────────────────────────────────────────────────
PROJECT_NAME = 'Bio Env'
RUN_NAME     = 'Run 1'
log_wandb    = False   # set True to enable wandb logging

if log_wandb and WANDB_AVAILABLE:
    print(f'Logging to wandb: project={PROJECT_NAME}  run={RUN_NAME}  time={curr_date_time}')
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)
else:
    print('WandB logging disabled.')

WandB logging disabled.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
FOOD_FOLDER = 'foods_dataset'
PLOT_DIR    = 'plots'
MENU_SIZE   = 2
SEED        = 0

# ── Sleep / wake cycle ────────────────────────────────────────────────────
# 480 awake steps × 2 min/step = 960 min = 16 hrs awake
# 240 sleep steps × 2 min/step =  480 min = 8 hrs asleep
# 1 cycle = 720 steps; 7 cycles (1 week) = 5040 steps
AWAKE_STEPS = 480
SLEEP_STEPS = 240
NUM_CYCLES  = 7          # one full week
MAX_STEPS   = NUM_CYCLES * (AWAKE_STEPS + SLEEP_STEPS)   # 20 160

os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Episode length: {MAX_STEPS} steps  ({NUM_CYCLES} cycles)')

Episode length: 5040 steps  (7 cycles)


In [4]:
# ── Environment ───────────────────────────────────────────────────────────
env = FoodEnv(
    food_folder=FOOD_FOLDER,
    num_foods=MENU_SIZE,
    max_steps=MAX_STEPS,
    one_hot_embedding=True,
    seed=SEED,
    consumption_threshold=0.1,    # amounts below this → zero (no absorption)
    awake_steps_per_cycle=AWAKE_STEPS,
    sleep_steps_per_cycle=SLEEP_STEPS,
)

print('\n── Environment summary ──────────────────────────────────────')
print(f'  Food items             : {env.num_items}')
print(f'  Nutrients              : {env.num_nutrients}  {env.nutrient_names}')
print(f'  Observation state dim  : {env.state_dim}  (nutrients + is_awake + time_in_cycle)')
print(f'  Menu size              : {env.num_foods}  → action shape {env.action_space.shape}')
print(f'  Max steps              : {env.max_steps}')
print(f'  Awake steps / cycle    : {env.awake_steps}')
print(f'  Sleep steps / cycle    : {env.sleep_steps}')
print(f'  Cycle length           : {env.cycle_length}')
print(f'  Consumption threshold  : {env.consumption_threshold}')
print(f'  Target (normed)        : {env._norm_targets}')
print(f'  Target low             : {env._norm_target_low}')
print(f'  Target high            : {env._norm_target_high}')
print()
print(env.nutrient_norm_summary().to_string(index=False))
print('─────────────────────────────────────────────────────────────')

[FoodEnv] Loading  'glucose'  from  'foods_dataset\serum_glucose.csv'
           min=0.0000  max=153.4313   foods=28   time_points=500
[FoodEnv] Loading  'peptides'  from  'foods_dataset\small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=28   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'foods_dataset\fatty_acids_absorbed.csv'
           min=0.0000  max=0.0049   foods=28   time_points=500
[FoodEnv] Ready — 28 foods | 3 time-series nutrients | 0 cumulative nutrients

── Environment summary ──────────────────────────────────────
  Food items             : 28
  Nutrients              : 3  ['glucose', 'peptides', 'fatty_acids']
  Observation state dim  : 5  (nutrients + is_awake + time_in_cycle)
  Menu size              : 2  → action shape (2,)
  Max steps              : 5040
  Awake steps / cycle    : 480
  Sleep steps / cycle    : 240
  Cycle length           : 720
  Consumption threshold  : 0.1
  Target (normed)        : [0.65175766 0.5586592  0.06804124]


In [5]:
# ── Agent ─────────────────────────────────────────────────────────────────
agent = PPOAgent(env, device='cpu', shared=False, seed=42, limit_delta=1.0)

In [6]:
# ── Train ─────────────────────────────────────────────────────────────────
NUM_EPISODES = 750
print(f'Starting training: {NUM_EPISODES} episodes …')

returns, consumption = agent.train(
    num_episodes=NUM_EPISODES,
    rollout_steps=512,
    ppo_epochs=4,
    minibatch_size=64,
    log_every_episodes=50,
    printing=True,
    actor_lr=1e-4,
    critic_lr=1e-3,
    log_wandb=log_wandb,
)

print(f'\nDone.')
print(f'  Final rolling avg return      (last 50): {np.mean(returns[-50:]):.3f}')
print(f'  Final rolling avg consumption (last 50): {np.mean(consumption[-50:]):.3f}')

Starting training: 750 episodes …


KeyboardInterrupt: 

In [ ]:
# ── Greedy evaluation episode ─────────────────────────────────────────────
print('\nRunning greedy evaluation episode …')
memory, episode_df = agent.generate_episode(log_wandb=log_wandb, episode_idx=0)

print(f'  Episode length      : {len(episode_df)} steps')
print(f'  Total reward        : {episode_df["reward"].sum():.2f}')
print(f'  Awake steps         : {episode_df["is_awake"].sum()}')
print(f'  Sleep steps         : {(~episode_df["is_awake"]).sum()}')
print(f'  Total consumption   : {episode_df["total_consumption"].sum():.2f}')
print(f'  Cycles completed    : {episode_df["cycle_number"].max() + 1}')
episode_df.head(10)

In [ ]:
# ── Rebuild arrays for plotting ───────────────────────────────────────────
T     = len(episode_df)
n     = env.num_nutrients
names = env.nutrient_names

# State trajectory (T+1, N) — last row repeated as terminal placeholder
states = np.vstack([
    episode_df[list(names)].values,
    episode_df[list(names)].iloc[[-1]].values,
])

rewards      = episode_df['reward'].values
distances    = episode_df['distance'].values
is_awake_arr = episode_df['is_awake'].values        # (T,) bool

amounts_arr = np.stack(
    [episode_df[f'amount_slot_{i}'].values for i in range(env.num_foods)],
    axis=1,
)  # (T, num_foods)

target_norm = env._norm_targets
norm_low    = env._norm_target_low
norm_high   = env._norm_target_high

# Unnormalise
unnorm_states  = np.zeros_like(states)
unnorm_targets = np.zeros(n, dtype=np.float32)
unnorm_low     = np.zeros(n, dtype=np.float32)
unnorm_high    = np.zeros(n, dtype=np.float32)
for i, nm in enumerate(names):
    v_min = env._nutrient_mins[nm]
    v_max = env._nutrient_maxs[nm]
    rng   = v_max - v_min
    unnorm_states[:, i] = states[:, i] * rng + v_min
    unnorm_targets[i]   = target_norm[i] * rng + v_min
    unnorm_low[i]       = norm_low[i]   * rng + v_min
    unnorm_high[i]      = norm_high[i]  * rng + v_min

t_axis = np.arange(T + 1)    # length T+1 (matches states)
t_step = np.arange(1, T + 1) # length T   (matches rewards / distances)

# Sleep windows for shading (list of (start, end) step tuples)
sleep_windows = env.sleep_windows()
print(f'Sleep windows in episode: {len(sleep_windows)}')

In [ ]:
# ── Helper: shade sleep windows on an axis ────────────────────────────────
def shade_sleep(ax, windows, t_max, label_first=True):
    """Shade each sleep window in light navy on ax."""
    for k, (ws, we) in enumerate(windows):
        ax.axvspan(ws, min(we, t_max), alpha=0.10, color='navy',
                   label='Sleep' if (k == 0 and label_first) else '')

In [ ]:
# ── Consolidated evaluation figure ───────────────────────────────────────
# Layout:
#   Row 0        : training curve (full width)
#   Rows 1…N     : normalised | unnormalised state per nutrient
#   Row N+1      : reward | L1 distance
#   Row N+2      : consumption heatmap | signed error heatmap
# Sleep windows are shaded in light navy on all time-series panels.

SKIP_EPS = 100   # skip noisy early episodes in training curve
WINDOW   = 50   # rolling average window

n_rows = n + 3
fig = plt.figure(figsize=(14, 4.5 * n_rows))
gs  = gridspec.GridSpec(n_rows, 2, figure=fig, hspace=0.5, wspace=0.35)

# ── Row 0: training curve ─────────────────────────────────────────────────
ax_tr        = fig.add_subplot(gs[0, :])
plot_returns  = returns[SKIP_EPS:]
plot_episodes = np.arange(SKIP_EPS, len(returns))
rolling       = np.convolve(plot_returns, np.ones(WINDOW) / WINDOW, mode='valid')
roll_x        = np.arange(SKIP_EPS + WINDOW - 1, len(returns))
ax_tr.plot(plot_episodes, plot_returns, alpha=0.3, color='steelblue', label='Episode return')
ax_tr.plot(roll_x, rolling, color='steelblue', lw=2, label=f'Rolling avg ({WINDOW})')
max_ret = float(np.max(returns))
max_ep  = int(np.argmax(returns))
ax_tr.scatter([max_ep], [max_ret], color='crimson', zorder=5, s=60)
ax_tr.annotate(f'Max: {max_ret:.2f}', xy=(max_ep, max_ret),
               xytext=(12, -18), textcoords='offset points',
               color='crimson', fontsize=9,
               arrowprops=dict(arrowstyle='->', color='crimson', lw=1.2))
ax_tr.set_xlabel('Episode')
ax_tr.set_ylabel('Return')
ax_tr.set_title(f'PPO Training Curve  (first {SKIP_EPS} eps omitted)')
ax_tr.legend(fontsize=9)
ax_tr.grid(True, alpha=0.3)

# ── Rows 1…N: nutrient states ─────────────────────────────────────────────
for i, nm in enumerate(names):
    row = i + 1

    # Normalised
    ax_n = fig.add_subplot(gs[row, 0])
    ax_n.axhspan(norm_low[i], norm_high[i], alpha=0.15, color='crimson',
                 label=f'Target zone [{norm_low[i]:.3f}, {norm_high[i]:.3f}]')
    ax_n.axhline(target_norm[i], ls='--', color='crimson', lw=1.0, alpha=0.5)
    ax_n.plot(t_axis, states[:, i], color='steelblue', lw=1.5, label='Agent state')
    ref_n = np.clip(states[:, i], norm_low[i], norm_high[i])
    ax_n.fill_between(t_axis, states[:, i], ref_n, alpha=0.12, color='steelblue')
    shade_sleep(ax_n, sleep_windows, T)
    ax_n.set_title(f'{nm}  —  normalised', fontsize=10)
    ax_n.set_xlabel('Step')
    ax_n.set_ylabel('Level (normalised)')
    ax_n.legend(fontsize=8)
    ax_n.grid(True, alpha=0.3)

    # Unnormalised
    ax_u = fig.add_subplot(gs[row, 1])
    ax_u.axhspan(unnorm_low[i], unnorm_high[i], alpha=0.15, color='crimson',
                 label=f'Target zone [{unnorm_low[i]:.4g}, {unnorm_high[i]:.4g}]')
    ax_u.axhline(unnorm_targets[i], ls='--', color='crimson', lw=1.0, alpha=0.5)
    ax_u.plot(t_axis, unnorm_states[:, i], color='darkorange', lw=1.5, label='Agent state')
    ref_u = np.clip(unnorm_states[:, i], unnorm_low[i], unnorm_high[i])
    ax_u.fill_between(t_axis, unnorm_states[:, i], ref_u, alpha=0.12, color='darkorange')
    shade_sleep(ax_u, sleep_windows, T)
    ax_u.set_title(f'{nm}  —  raw units', fontsize=10)
    ax_u.set_xlabel('Step')
    ax_u.set_ylabel('Level (raw units)')
    ax_u.legend(fontsize=8)
    ax_u.grid(True, alpha=0.3)

# ── Row N+1: reward | distance ────────────────────────────────────────────
ax_r = fig.add_subplot(gs[n + 1, 0])
ax_r.plot(t_step, rewards, color='darkorange', lw=1.5)
ax_r.axhline(0, color='grey', lw=0.8, ls='--')
shade_sleep(ax_r, sleep_windows, T)
ax_r.set_ylabel('Reward')
ax_r.set_xlabel('Step')
ax_r.set_title('Reward per step')
ax_r.grid(True, alpha=0.3)

ax_d = fig.add_subplot(gs[n + 1, 1])
ax_d.plot(t_step, distances, color='teal', lw=1.5)
shade_sleep(ax_d, sleep_windows, T)
ax_d.set_ylabel('L1 distance to target')
ax_d.set_xlabel('Step')
ax_d.set_title('Distance to target')
ax_d.grid(True, alpha=0.3)

# ── Row N+2: consumption heatmap | signed error heatmap ───────────────────
# LEFT: consumption amounts per slot over time.
# Sleep steps show as zero (env blocked intake) — visually confirms the
# agent learned not to rely on food during sleep.
ax_ch = fig.add_subplot(gs[n + 2, 0])
im_c = ax_ch.imshow(
    amounts_arr.T,   # (num_foods, T)
    aspect='auto', cmap='YlOrRd', vmin=0.0, vmax=1.0,
    interpolation='nearest',
)
ax_ch.set_yticks(range(env.num_foods))
ax_ch.set_yticklabels([f'slot {i}' for i in range(env.num_foods)])
ax_ch.set_xlabel('Step')
ax_ch.set_title('Consumption amounts  (slot × time)')
plt.colorbar(im_c, ax=ax_ch, label='Amount [0, 1]')

# RIGHT: signed range error (0 inside zone, <0 below, >0 above)
below = np.minimum(0.0, states - norm_low[None, :])
above = np.maximum(0.0, states - norm_high[None, :])
error = below + above
ax_h  = fig.add_subplot(gs[n + 2, 1])
im    = ax_h.imshow(
    error.T, aspect='auto', cmap='RdBu_r',
    vmin=-np.abs(error).max(), vmax=np.abs(error).max(),
    interpolation='nearest',
)
ax_h.set_yticks(range(n))
ax_h.set_yticklabels(names)
ax_h.set_xlabel('Step')
ax_h.set_title('Signed error  (state − target, normalised)')
plt.colorbar(im, ax=ax_h, label='Error')

fig.suptitle('PPO Agent — Evaluation Summary', fontsize=14, y=1.005)
fig.savefig(os.path.join(PLOT_DIR, 'evaluation_summary.png'), bbox_inches='tight')
print('Saved evaluation_summary.png')
plt.show()

In [ ]:
# ── Digestion dynamics (env built-in, with sleep shading) ─────────────────
fig_dig = env.plot_consumption()
fig_dig.savefig(os.path.join(PLOT_DIR, 'digestion_dynamics.png'), bbox_inches='tight')
print('Saved digestion_dynamics.png')
plt.show()

In [ ]:
# ── Per-day breakdown ─────────────────────────────────────────────────────
# Split episode_df by cycle_number and show mean reward and consumption per day.
daily = episode_df.groupby('cycle_number').agg(
    mean_reward=('reward', 'mean'),
    total_consumption=('total_consumption', 'sum'),
    awake_steps=('is_awake', 'sum'),
    sleep_steps=('is_awake', lambda x: (~x).sum()),
).reset_index()
print('Per-cycle (day) summary:')
display(daily)

In [ ]:
# ── Per-day mean reward bar chart ─────────────────────────────────────────
fig_d, ax_d = plt.subplots(figsize=(10, 4))
ax_d.bar(daily['cycle_number'], daily['mean_reward'], color='steelblue', alpha=0.8)
ax_d.axhline(0, color='grey', lw=0.8, ls='--')
ax_d.set_xlabel('Cycle (day)')
ax_d.set_ylabel('Mean reward per step')
ax_d.set_title('Mean reward per cycle')
ax_d.set_xticks(daily['cycle_number'])
ax_d.grid(True, axis='y', alpha=0.3)
fig_d.savefig(os.path.join(PLOT_DIR, 'per_day_reward.png'), bbox_inches='tight')
print('Saved per_day_reward.png')
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────
os.makedirs('checkpoints', exist_ok=True)
agent.save_model('checkpoints/ppo_agent', log_wandb=log_wandb)
print('Model saved to checkpoints/')

In [ ]:
# ── (Optional) finish wandb run ───────────────────────────────────────────
if log_wandb and WANDB_AVAILABLE:
    wandb.finish()
    print('WandB run finished.')